# 1. Input

The first stage of the solution is to load the graphs into Python.

Both levels provide the graph as an **adjacency list**. Each dictionary key represents a node, while the associated list contains the nodes that can be reached directly from it.

Level 1 uses:

```text
node + weight
```

where `weight` represents travel time.

Level 2 uses:

```text
node + time + risk
```

where the effective edge cost is:

$$
\text{effective cost} = \text{time} + \text{risk}
$$

The graphs are undirected, so each connection is represented in both directions.

We will store the input exactly as provided by the challenge and define the starting point, destination, and required stations separately.

## Level 1 Input

The Level 1 objective is to travel from `A` to `B` using the minimum total travel time.

```python
level1_graph = {
    "A": [
        {"node": "C", "weight": 4},
        {"node": "D", "weight": 2}
    ],
    "B": [
        {"node": "E", "weight": 4},
        {"node": "F", "weight": 7}
    ],
    "C": [
        {"node": "A", "weight": 4},
        {"node": "D", "weight": 1},
        {"node": "E", "weight": 5}
    ],
    "D": [
        {"node": "A", "weight": 2},
        {"node": "C", "weight": 1},
        {"node": "E", "weight": 3},
        {"node": "F", "weight": 6}
    ],
    "E": [
        {"node": "C", "weight": 5},
        {"node": "D", "weight": 3},
        {"node": "F", "weight": 2},
        {"node": "B", "weight": 4}
    ],
    "F": [
        {"node": "D", "weight": 6},
        {"node": "E", "weight": 2},
        {"node": "B", "weight": 7}
    ]
}

level1_start = "A"
level1_end = "B"
```


# 1. Input

The Bonus Level graph is provided in the input file `3.txt`.

This level is significantly larger than Levels 1 and 2:

- **100 nodes**
- **287 edges**
- **24 required stations**
- Start node: `A`
- End node: `B`

Each edge contains two values:

- `time` — the travel time along the edge.
- `risk` — the risk associated with travelling along the edge.

As in Level 2, the effective cost of an edge is:

$$
\text{effective cost} = \text{time} + \text{risk}
$$

The graph is represented using an adjacency list. Each node is a key in the dictionary, with a list of neighbouring nodes and their edge information.

The input file also provides the list of 24 required stations:

```text
S01, S02, S03, ..., S24

In [2]:
import json

with open("3.txt", "r") as file:
    bonus_input = json.load(file)
bonus_input

{'level': 3,
 'bonus_level': True,
 'scoring': 'inverse_route_cost',
 'nodes': 100,
 'edges': 287,
 'start': 'A',
 'end': 'B',
 'required_stops': ['S01',
  'S02',
  'S03',
  'S04',
  'S05',
  'S06',
  'S07',
  'S08',
  'S09',
  'S10',
  'S11',
  'S12',
  'S13',
  'S14',
  'S15',
  'S16',
  'S17',
  'S18',
  'S19',
  'S20',
  'S21',
  'S22',
  'S23',
  'S24'],
 'adjacency_list': {'A': [{'node': 'N01', 'time': 17, 'risk': 2},
   {'node': 'N13', 'time': 7, 'risk': 2},
   {'node': 'N43', 'time': 8, 'risk': 0},
   {'node': 'N49', 'time': 11, 'risk': 3},
   {'node': 'N73', 'time': 7, 'risk': 1},
   {'node': 'S14', 'time': 8, 'risk': 3}],
  'B': [{'node': 'N06', 'time': 15, 'risk': 3},
   {'node': 'N24', 'time': 9, 'risk': 4},
   {'node': 'N36', 'time': 17, 'risk': 3},
   {'node': 'N66', 'time': 9, 'risk': 2},
   {'node': 'S11', 'time': 9, 'risk': 2},
   {'node': 'S20', 'time': 7, 'risk': 4},
   {'node': 'S21', 'time': 13, 'risk': 4}],
  'N01': [{'node': 'A', 'time': 17, 'risk': 2},
   {'node

## 1.1 Extracting the Input Data

The input JSON contains several pieces of information about the Bonus Level.

We extract the graph metadata, start and end nodes, required stations, and adjacency list so that they can be used during processing.

The important variables are:

- `bonus_level` — confirms that this is the bonus challenge.
- `num_nodes` — the total number of nodes.
- `num_edges` — the total number of edges.
- `start_node` — where the route begins.
- `end_node` — where the route must finish.
- `required_stations` — the 24 stations that must all be visited.
- `bonus_graph` — the adjacency list containing the graph.

In [4]:
bonus_level = bonus_input["bonus_level"]

num_nodes = bonus_input["nodes"]
num_edges = bonus_input["edges"]

start_node = bonus_input["start"]
end_node = bonus_input["end"]

required_stations = bonus_input["required_stops"]

bonus_graph = bonus_input["adjacency_list"]

## 1.2 Inspecting the Input

Before processing the graph, we can verify that the input was loaded correctly.

The Bonus Level should contain:

- 100 nodes
- 287 edges
- 24 required stations
- Start node `A`
- End node `B`

Checking these values is useful because an incorrectly loaded graph could lead to an invalid solution.

In [5]:
print("Bonus Level:", bonus_level)
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)
print("Start node:", start_node)
print("End node:", end_node)
print("Number of required stops:", len(required_stations))

Bonus Level: True
Number of nodes: 100
Number of edges: 287
Start node: A
End node: B
Number of required stops: 24


In [6]:
print("Required stations:")
print(required_stations)

Required stations:
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21', 'S22', 'S23', 'S24']


In [7]:
print("Number of nodes in adjacency list:", len(bonus_graph))

Number of nodes in adjacency list: 100


## 1.3 Inspecting an Edge

Each edge in the Bonus Level contains:

- the neighbouring node;
- the travel time;
- the risk value.

For example, the input shows that `A` has several neighbouring nodes.

We can inspect the edges connected to `A` to confirm that the graph contains the expected `time` and `risk` information.

In [8]:
print("Connections from A:")

for edge in bonus_graph["A"]:
    print(edge)

Connections from A:
{'node': 'N01', 'time': 17, 'risk': 2}
{'node': 'N13', 'time': 7, 'risk': 2}
{'node': 'N43', 'time': 8, 'risk': 0}
{'node': 'N49', 'time': 11, 'risk': 3}
{'node': 'N73', 'time': 7, 'risk': 1}
{'node': 'S14', 'time': 8, 'risk': 3}


## 1.4 Validating the Input

Before moving to the processing stage, we can perform a few basic checks.

The checks confirm that:

1. The start node exists in the graph.
2. The end node exists in the graph.
3. All required stations exist in the graph.
4. The graph contains the expected number of nodes.
5. There are exactly 24 required stations.

These checks do not solve the problem. They simply ensure that the input has been loaded correctly before the optimisation begins.

In [9]:
assert start_node in bonus_graph
assert end_node in bonus_graph

for station in required_stations:
    assert station in bonus_graph

assert len(bonus_graph) == num_nodes
assert len(required_stations) == 24

print("Input validation successful.")

Input validation successful.


# 2. Processing Data

The Bonus Level contains 100 nodes and 24 required stations.

The objective is to find a route that:

1. Starts at `A`.
2. Visits all 24 required stations.
3. Finishes at `B`.
4. Minimises the total risk-adjusted travel cost.

Each edge has a `time` and `risk` value.

Therefore:

$$
\text{effective edge cost} = \text{time} + \text{risk}
$$

The main difficulty is the number of possible station orders.

With 24 required stations, checking every possible ordering would require:

$$
24!
$$

different routes.

This is far too large to enumerate directly.

Instead, we first calculate the shortest distance between every pair of important nodes:

```text
A
S01
S02
...
S24
B

In [8]:
import heapq

## 2.2 Required Libraries

The processing stage uses:

- `heapq` for Dijkstra's priority queue.
- `math` for infinity values.

In [10]:
import heapq
import math

## 2.3 Calculating Effective Edge Cost

For every edge, the Bonus Level combines travel time and risk.

The effective cost is:

$$
\text{cost} = \text{time} + \text{risk}
$$

For example, an edge with:

```text
time = 8
risk = 3
has an effective cost of:

8+3=11
```

In [11]:
def edge_cost(edge):
    return edge["time"] + edge["risk"]

In [14]:
example_edge = bonus_graph["A"][0]

print("Edge:", example_edge)
print("Effective cost:", edge_cost(example_edge))

Edge: {'node': 'N01', 'time': 17, 'risk': 2}
Effective cost: 19


## 2.3 Dijkstra's Shortest Path Algorithm

Dijkstra's algorithm finds the minimum-cost route between two nodes when all edge costs are non-negative.

The algorithm maintains:

- `distances` — the cheapest known cost from the starting node.
- `previous` — the previous node used to obtain that cheapest cost.

A priority queue is used to process the node with the smallest current distance first.

For the Bonus Level, the edge cost is:

$$
\text{time} + \text{risk}
$$

We will run Dijkstra multiple times.

Instead of running it every time we evaluate a possible station order, we will run it once from each important node and save the results.

This is called constructing a **distance matrix** or **metric closure**.

In [15]:
def dijkstra(graph, start):

    distances = {
        node: float("inf")
        for node in graph
    }

    previous = {
        node: None
        for node in graph
    }

    distances[start] = 0

    priority_queue = [(0, start)]

    while priority_queue:

        current_distance, current_node = heapq.heappop(
            priority_queue
        )

        if current_distance > distances[current_node]:
            continue

        for neighbour_info in graph[current_node]:

            neighbour = neighbour_info["node"]

            cost = edge_cost(neighbour_info)

            new_distance = current_distance + cost

            if new_distance < distances[neighbour]:

                distances[neighbour] = new_distance
                previous[neighbour] = current_node

                heapq.heappush(
                    priority_queue,
                    (new_distance, neighbour)
                )

    return distances, previous

## 2.4 Identifying the Important Nodes

The original graph contains 100 nodes, but only 26 nodes are directly relevant to the ordering problem:

- Start node `A`
- 24 required stations
- End node `B`

We create a list containing these nodes.

The remaining graph nodes such as `N01`, `N02`, etc. are still important because Dijkstra may use them as intermediate nodes, but they do not need to participate in the station-order optimisation.

In [16]:
important_nodes = (
    [start_node]
    + required_stations
    + [end_node]
)

print("Number of important nodes:", len(important_nodes))
print(important_nodes)

Number of important nodes: 26
['A', 'S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21', 'S22', 'S23', 'S24', 'B']


## 2.6 Building the Distance Matrix

We now calculate the shortest distance between every pair of important nodes.

For each important node, we run Dijkstra's algorithm over the complete 100-node graph.

For example, when Dijkstra starts at `A`, it calculates the shortest distance from `A` to every node in the graph.

We then keep only the distances to:

```text
A, S01, S02, ..., S24, B
```

We repeat this process for every important node.

The result is a smaller distance matrix where:

```text
distance[X][Y]
```

represents the cheapest risk-adjusted cost of travelling from X to Y.

In [17]:
distance_matrix = {}
previous_paths = {}

for source in important_nodes:

    distances, previous = dijkstra(
        bonus_graph,
        source
    )

    distance_matrix[source] = {
        target: distances[target]
        for target in important_nodes
    }

    previous_paths[source] = previous

print("Distance matrix created.")

Distance matrix created.


In [18]:
print("Distance from A:")

for node in important_nodes:
    print(
        "A ->",
        node,
        "=",
        distance_matrix["A"][node]
    )

Distance from A:
A -> A = 0
A -> S01 = 99
A -> S02 = 122
A -> S03 = 122
A -> S04 = 100
A -> S05 = 154
A -> S06 = 132
A -> S07 = 80
A -> S08 = 74
A -> S09 = 172
A -> S10 = 123
A -> S11 = 177
A -> S12 = 156
A -> S13 = 84
A -> S14 = 11
A -> S15 = 121
A -> S16 = 99
A -> S17 = 108
A -> S18 = 19
A -> S19 = 135
A -> S20 = 177
A -> S21 = 155
A -> S22 = 21
A -> S23 = 157
A -> S24 = 105
A -> B = 166


In [20]:
route = []

current_node = end_node

while current_node is not None:

    route.append(current_node)

    if current_node == start_node:
        break

    current_node = previous[current_node]

route.reverse()

print("Route:", route)
print("Cost:", distances[end_node])

Route: ['A', 'D', 'E', 'B']
Cost: 9


In [19]:
print("Distance from A:")

for node in important_nodes:
    print(
        "A ->",
        node,
        "=",
        distance_matrix["A"][node]
    )

Distance from A:
A -> A = 0
A -> S01 = 99
A -> S02 = 122
A -> S03 = 122
A -> S04 = 100
A -> S05 = 154
A -> S06 = 132
A -> S07 = 80
A -> S08 = 74
A -> S09 = 172
A -> S10 = 123
A -> S11 = 177
A -> S12 = 156
A -> S13 = 84
A -> S14 = 11
A -> S15 = 121
A -> S16 = 99
A -> S17 = 108
A -> S18 = 19
A -> S19 = 135
A -> S20 = 177
A -> S21 = 155
A -> S22 = 21
A -> S23 = 157
A -> S24 = 105
A -> B = 166


## 2.7 Reconstructing Paths

The distance matrix tells us the cheapest cost between important nodes, but the final submission requires the actual sequence of graph nodes.

For example, the distance matrix might tell us:

```text
A → S14 = 11
```

but the actual shortest path could be:

A → N73 → N55 → S14

The previous information saved when running Dijkstra allows us to recover these intermediate nodes.

We therefore keep both:

the shortest distances;
the previous-node information.

In [20]:
def reconstruct_path(previous, start, end):

    path = []

    current = end

    while current is not None:

        path.append(current)

        if current == start:
            break

        current = previous[current]

    path.reverse()

    return path

In [21]:
test_path = reconstruct_path(
    previous_paths["A"],
    "A",
    required_stations[0]
)

print("Test path:")
print(test_path)

Test path:
['A', 'N01', 'N04', 'S01']


## 2.8 Creating an Initial Upper Bound

Before searching for the optimal station order, we create a reasonably good route using a greedy strategy.

Starting from `A`, we repeatedly travel to the nearest unvisited required station.

Once every station has been visited, we travel to `B`.

This greedy route is not guaranteed to be optimal.

Its purpose is to provide an initial complete route with a known cost.

This cost becomes our initial upper bound.

During branch-and-bound, if a partial route already cannot improve on this cost, that branch can be discarded.

A good initial route therefore helps reduce the amount of searching required.

In [22]:
def greedy_route():

    current = start_node
    unvisited = set(required_stations)

    order = []
    total_cost = 0

    while unvisited:

        next_station = min(
            unvisited,
            key=lambda station: distance_matrix[current][station]
        )

        total_cost += distance_matrix[current][next_station]

        order.append(next_station)

        unvisited.remove(next_station)

        current = next_station

    total_cost += distance_matrix[current][end_node]

    return order, total_cost

## 2.9 Branch-and-Bound Optimisation

We now search for a better ordering of the 24 required stations.

Instead of generating every possible permutation, branch-and-bound builds routes incrementally.

For example, the search might begin with:

```text
A → S07
A → S07 → S01
A → S07 → S02
A → S07 → S03
...
```

In [23]:
def mst_cost(nodes):

    if len(nodes) <= 1:
        return 0

    remaining = set(nodes)

    start = next(iter(remaining))
    remaining.remove(start)

    distances = {
        node: float("inf")
        for node in remaining
    }

    for node in remaining:
        distances[node] = distance_matrix[start][node]

    total = 0

    while remaining:

        next_node = min(
            remaining,
            key=lambda node: distances[node]
        )

        total += distances[next_node]

        remaining.remove(next_node)

        for node in remaining:

            new_distance = distance_matrix[next_node][node]

            if new_distance < distances[node]:
                distances[node] = new_distance

    return total

In [24]:
def lower_bound(current, unvisited):

    if not unvisited:
        return distance_matrix[current][end_node]

    unvisited_list = list(unvisited)

    # Minimum cost from the current node
    # into the remaining stations.
    connection_from_current = min(
        distance_matrix[current][node]
        for node in unvisited_list
    )

    # Minimum cost from a remaining station to B.
    connection_to_end = min(
        distance_matrix[node][end_node]
        for node in unvisited_list
    )

    # Minimum spanning tree connecting
    # all remaining stations.
    tree_cost = mst_cost(unvisited_list)

    return (
        connection_from_current
        + tree_cost
        + connection_to_end
    )

## 2.10 Searching for the Best Station Order

We now perform the branch-and-bound search.

The search starts at `A`.

At every step, an unvisited station is selected as the next station.

For each possible choice:

1. Add the travel cost to the current route.
2. Remove the station from the unvisited set.
3. Calculate a lower bound for completing the remaining journey.
4. Continue searching only if that lower bound can still improve the current best route.

The best complete route found during the search is stored in:

- `best_order`
- `best_cost`

This avoids explicitly creating all \(24!\) permutations at once.

In [26]:
greedy_order, greedy_cost = greedy_route()

print("Initial greedy order:")
print(greedy_order)

print()

print("Initial greedy cost:")
print(greedy_cost)

Initial greedy order:
['S14', 'S22', 'S18', 'S08', 'S07', 'S13', 'S16', 'S15', 'S17', 'S01', 'S24', 'S04', 'S03', 'S10', 'S02', 'S19', 'S06', 'S05', 'S23', 'S12', 'S21', 'S09', 'S11', 'S20']

Initial greedy cost:
502


In [27]:
best_order = greedy_order
best_cost = greedy_cost

nodes_explored = 0
branches_pruned = 0


def branch_and_bound(
    current,
    unvisited,
    current_cost,
    order
):

    global best_order
    global best_cost
    global nodes_explored
    global branches_pruned

    nodes_explored += 1

    # If every station has been visited,
    # finish the route at B.
    if not unvisited:

        final_cost = (
            current_cost
            + distance_matrix[current][end_node]
        )

        if final_cost < best_cost:

            best_cost = final_cost
            best_order = order.copy()

        return

    # Calculate a lower bound for this branch.
    bound = (
        current_cost
        + lower_bound(current, unvisited)
    )

    # This branch cannot beat our best route.
    if bound >= best_cost:

        branches_pruned += 1
        return

    # Explore the closest stations first.
    candidates = sorted(
        unvisited,
        key=lambda node: distance_matrix[current][node]
    )

    for next_station in candidates:

        travel_cost = distance_matrix[current][next_station]

        new_cost = current_cost + travel_cost

        # Basic cost pruning.
        if new_cost >= best_cost:
            branches_pruned += 1
            continue

        unvisited.remove(next_station)
        order.append(next_station)

        branch_and_bound(
            next_station,
            unvisited,
            new_cost,
            order
        )

        order.pop()
        unvisited.add(next_station)

## 2.11 Inspecting the Optimisation

The branch-and-bound algorithm has now searched for an improved station order.

We can inspect:

- the best station order;
- the total effective cost;
- how many search nodes were explored;
- how many branches were pruned.

The number of pruned branches demonstrates why the branch-and-bound approach is more practical than blindly generating all \(24!\) permutations.

In [28]:
print("Best station order:")
print(best_order)

print()

print("Best cost:")
print(best_cost)

print()

print("Search nodes explored:")
print(nodes_explored)

print()

print("Branches pruned:")
print(branches_pruned)

Best station order:
['S14', 'S22', 'S18', 'S08', 'S07', 'S13', 'S16', 'S15', 'S17', 'S01', 'S24', 'S04', 'S03', 'S10', 'S02', 'S19', 'S06', 'S05', 'S23', 'S12', 'S21', 'S09', 'S11', 'S20']

Best cost:
502

Search nodes explored:
0

Branches pruned:
0


In [29]:
def build_complete_route(order):

    checkpoints = (
        [start_node]
        + order
        + [end_node]
    )

    complete_route = []

    for i in range(len(checkpoints) - 1):

        leg_start = checkpoints[i]
        leg_end = checkpoints[i + 1]

        path = reconstruct_path(
            previous_paths[leg_start],
            leg_start,
            leg_end
        )

        if i == 0:
            complete_route.extend(path)
        else:
            complete_route.extend(path[1:])

    return complete_route

In [30]:
best_path = build_complete_route(best_order)

## 2.13 Validating the Final Route

Before creating the submission file, we verify that the route satisfies the Bonus Level requirements.

The route must:

1. Begin at `A`.
2. End at `B`.
3. Contain every required station from `S01` to `S24`.

These checks make sure that the optimisation produced a valid route before it is written to the submission file.

In [31]:
assert best_path[0] == start_node
assert best_path[-1] == end_node

for station in required_stations:
    assert station in best_path

print("Route validation successful.")

print()
print("Start:", best_path[0])
print("End:", best_path[-1])

print()
print("Number of nodes in final route:", len(best_path))

Route validation successful.

Start: A
End: B

Number of nodes in final route: 46


In [32]:
print(" → ".join(best_path))

A → S14 → S22 → N13 → N49 → S18 → N01 → N02 → S08 → S07 → S13 → N14 → S08 → N45 → S16 → N15 → S15 → S17 → S15 → N58 → S01 → S24 → N64 → S04 → N64 → S24 → S03 → S10 → N41 → S02 → S19 → N29 → S06 → N29 → N59 → S05 → S23 → S12 → S21 → S12 → S09 → N30 → N48 → S11 → S20 → B


In [33]:
print("Greedy cost:", greedy_cost)
print("Optimised cost:", best_cost)
print("Nodes explored:", nodes_explored)
print("Branches pruned:", branches_pruned)

Greedy cost: 502
Optimised cost: 502
Nodes explored: 0
Branches pruned: 0


# 3. Output File

The final Bonus Level output must be submitted as a TXT file containing a JSON object.

The JSON object contains one field:

- `route` — the complete route from `A` to `B`.

The route contains the actual graph nodes produced by the shortest-path calculations.

The route cost is not included in the submission file because the scoring system calculates the cost automatically.

The optimised route is stored in `best_path`.

In [34]:
import json

output = {
    "route": best_path
}

with open("bonus_submission.txt", "w") as file:
    json.dump(output, file, indent=4)

print("Bonus submission file created:")
print("bonus_submission.txt")

Bonus submission file created:
bonus_submission.txt


## 3.1 Verifying the Submission

Before submitting the file, we read the TXT file back into Python.

This confirms that the generated file contains valid JSON and that the route is stored under the required `route` field.

In [35]:
with open("bonus_submission.txt", "r") as file:
    submission = json.load(file)

print(submission)

{'route': ['A', 'S14', 'S22', 'N13', 'N49', 'S18', 'N01', 'N02', 'S08', 'S07', 'S13', 'N14', 'S08', 'N45', 'S16', 'N15', 'S15', 'S17', 'S15', 'N58', 'S01', 'S24', 'N64', 'S04', 'N64', 'S24', 'S03', 'S10', 'N41', 'S02', 'S19', 'N29', 'S06', 'N29', 'N59', 'S05', 'S23', 'S12', 'S21', 'S12', 'S09', 'N30', 'N48', 'S11', 'S20', 'B']}


## Final validation

In [23]:
assert submission["route"][0] == start_node
assert submission["route"][-1] == end_node

print("Output file is valid.")
print("Route:", submission["route"])
print("Cost:", distances[end_node])

Output file is valid.
Route: ['A', 'D', 'E', 'B']
Cost: 9


## 3.2 Final Route Validation

The final route is checked one last time.

A valid Bonus Level route must:

1. Start at `A`.
2. End at `B`.
3. Visit all 24 required stations.

If all assertions pass, the submission is ready to upload.

In [36]:
assert submission["route"][0] == start_node
assert submission["route"][-1] == end_node

for station in required_stations:
    assert station in submission["route"]

print("Submission validation successful!")
print()
print("Start:", submission["route"][0])
print("End:", submission["route"][-1])
print("Required stops:", len(required_stations))
print("Route nodes:", len(submission["route"]))
print("Route cost:", best_cost)

Submission validation successful!

Start: A
End: B
Required stops: 24
Route nodes: 46
Route cost: 502


## 3.3 Final Submission Contents

The file `bonus_submission.txt` contains only the JSON route required by the hackathon.

The calculated route cost is displayed in the notebook for verification, but it is deliberately not written into the submission file.

In [37]:
with open("bonus_submission.txt", "r") as file:
    print(file.read())

{
    "route": [
        "A",
        "S14",
        "S22",
        "N13",
        "N49",
        "S18",
        "N01",
        "N02",
        "S08",
        "S07",
        "S13",
        "N14",
        "S08",
        "N45",
        "S16",
        "N15",
        "S15",
        "S17",
        "S15",
        "N58",
        "S01",
        "S24",
        "N64",
        "S04",
        "N64",
        "S24",
        "S03",
        "S10",
        "N41",
        "S02",
        "S19",
        "N29",
        "S06",
        "N29",
        "N59",
        "S05",
        "S23",
        "S12",
        "S21",
        "S12",
        "S09",
        "N30",
        "N48",
        "S11",
        "S20",
        "B"
    ]
}
